In [2]:
import pandas as pd
import numpy as np 
from sklearn.preprocessing import LabelEncoder, StandardScaler, MinMaxScaler

In [4]:
df = pd.read_csv('../data/retail_sales.csv')
df['Date'] = pd.to_datetime(df['Date'])

In [5]:
#date parts

df['Month'] =df['Date'].dt.month
df['DayOfWeek'] =df['Date'].dt.dayofweek
df['IsWeekend'] = df['Date'].dt.dayofweek>=5
df['Quarter'] = df['Date'].dt.quarter

In [7]:
#bins
df['Age_Group'] = pd.cut(df['Age'],
bins=[0,25,35,50,100],
labels=['Young','Adult','Middle','Senior']
)

In [8]:
#ratio
df['Avg_Per_Unit'] = df['Total Amount']/ df['Quantity']

In [9]:
#category mean spend
df['Category_Avg']= df.groupby('Product Category')['Total Amount'].transform('mean')

In [10]:
#encoding
le = LabelEncoder()
df['Gender_Encoded'] = le.fit_transform(df['Gender'])
df_encoded = pd.get_dummies(df, columns=['Product Category'])

In [12]:
#scaling
scaler = StandardScaler()
df['Amount_Scaled'] = scaler.fit_transform(df[['Total Amount']])

minmax= MinMaxScaler()
df['Age_Scaled'] = minmax.fit_transform(df[['Age']])

In [ ]:
print("new columns added:")
print(df.columns.tolist())
print("\nSample rows:")
print(df[['Age', 'Age_Group','Age_Scaled',
          'Total Amount','Amount_Scaled',
          'Gender','Gender_Encoded',
          'Month','IsWeekend']].head(10))

new columns added:
['Transaction ID', 'Date', 'Customer ID', 'Gender', 'Age', 'Product Category', 'Quantity', 'Price per Unit', 'Total Amount', 'Month', 'DayOfWeek', 'IsWeekend', 'Quarter', 'Age_Group', 'Avg_Per_Unit', 'Category_Avg', 'Gender_Encoded', 'Amount_Scaled', 'Age_Scaled']

Sample rows:
     Age Age_Group  Age_Scaled  Total Amount  Amount_Scaled
0     34     Adult    0.347826           150      -0.546704
1     26     Adult    0.173913          1000       0.971919
2     50    Middle    0.695652            30      -0.761098
3     37    Middle    0.413043           500       0.078611
4     30     Adult    0.260870           100      -0.636035
..   ...       ...         ...           ...            ...
995   62    Senior    0.956522            50      -0.725366
996   52    Senior    0.739130            90      -0.653901
997   23     Young    0.108696           100      -0.636035
998   36    Middle    0.391304           150      -0.546704
999   47    Middle    0.630435           1

In [14]:
# reusable function
def engineer_features(df):
    df = df.copy()
    df['Date']       = pd.to_datetime(df['Date'])
    df['Month']      = df['Date'].dt.month
    df['DayOfWeek']  = df['Date'].dt.dayofweek
    df['IsWeekend']  = df['Date'].dt.dayofweek >= 5
    df['Quarter']    = df['Date'].dt.quarter
    df['Age_Group']  = pd.cut(df['Age'],
                          bins=[0, 25, 35, 50, 100],
                          labels=['Young', 'Adult', 'Middle', 'Senior'])
    df['Avg_Per_Unit']  = df['Total Amount'] / df['Quantity']
    df['Category_Avg']  = df.groupby('Product Category')['Total Amount'].transform('mean')
    return df

In [16]:
df_featured = engineer_features(df)
print(df_featured.shape)

(1000, 19)


MERGING
──────────────────────────────────────────
inner  → only rows that match in BOTH tables
left   → all rows from left + matches from right
right  → all rows from right + matches from left
outer  → everything from both tables

pd.merge(df1, df2, left_on='col1', right_on='col2', how='inner')

GROUPBY
──────────────────────────────────────────
.agg()        → summarizes into fewer rows (for reports)
.transform()  → keeps same number of rows (for new columns)

df.groupby('Category')['Amount'].agg('mean')       → summary table
df.groupby('Category')['Amount'].transform('mean') → new column

FEATURE ENGINEERING
──────────────────────────────────────────
Ratios    → col1 / col2          (e.g. Amount / Quantity)
Bins      → pd.cut()             (e.g. Age → Young/Adult/Senior)
Date parts → .dt.month           (e.g. Date → Month, DayOfWeek)

ENCODING (categories → numbers)
──────────────────────────────────────────
LabelEncoder   → ordered categories  (Small < Medium < Large)
get_dummies()  → unordered categories (Beauty, Clothing, Electronics)

SCALING (make all numbers fair)
──────────────────────────────────────────
StandardScaler → centers around 0     (use for most ML models)
MinMaxScaler   → squishes to 0 and 1  (use for neural networks)

WHEN TO USE WHICH?
──────────────────────────────────────────
Categories with order    → LabelEncoder
Categories without order → get_dummies
Normal ML models         → StandardScaler
Neural networks          → MinMaxScaler